In [ ]:
import mediapipe as mp
import numpy as np
import cv2
import socket
from PIL import Image
import io
import struct
import time
import threading

class global_vars:
    KILL_THREADS = False
    DEBUG = True
    WEBCAM_INDEX = 0
    USE_CUSTOM_CAM_SETTINGS = False
    FPS = 10
    WIDTH = 320
    HEIGHT = 240
    MODEL_COMPLEXITY = 2
    HOST = socket.gethostbyname(socket.gethostname()) 
    PORT = 3004
    BROADCAST_PORT = 37021
    FOCAL_LENGTH = 192  

def broadcast_ip():
    broadcast_message = f"{global_vars.HOST}"
    broadcast_address = ('<broadcast>', global_vars.BROADCAST_PORT)
    broadcast_socket = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    broadcast_socket.setsockopt(socket.SOL_SOCKET, socket.SO_BROADCAST, 1)
    
    while not global_vars.KILL_THREADS:
        try:
            broadcast_socket.sendto(broadcast_message.encode('utf-8'), broadcast_address)
            time.sleep(1)  
        except Exception as e:
            print("Broadcast error:", e)
            break
    broadcast_socket.close()

def compute_real_world_landmarks(world_landmarks, image_landmarks, image_shape):
    try:
        frame_height, frame_width, _ = image_shape
        focal_length = global_vars.FOCAL_LENGTH
        center = (frame_width / 2, frame_height / 2)
        camera_matrix = np.array(
            [[focal_length, 0, center[0]],
             [0, focal_length, center[1]],
             [0, 0, 1]], dtype="double"
        )
        distortion = np.zeros((4, 1))

        success, rotation_vector, translation_vector = cv2.solvePnP(
            objectPoints=world_landmarks, 
            imagePoints=image_landmarks, 
            cameraMatrix=camera_matrix, 
            distCoeffs=distortion,
            flags=cv2.SOLVEPNP_SQPNP
        )
        if not success:
            return world_landmarks

        transformation = np.eye(4)
        transformation[0:3, 3] = translation_vector.squeeze()

        model_points_hom = np.concatenate((world_landmarks, np.ones((33, 1))), axis=1)
        world_points = model_points_hom.dot(np.linalg.inv(transformation).T)

        return world_points
    except AttributeError:
        print("Attribute Error: shouldn't happen frequently")
        return world_landmarks

def process_pose(conn):
    mp_pose = mp.solutions.pose

    with mp_pose.Pose(
        min_detection_confidence=0.80,
        min_tracking_confidence=0.5,
        model_complexity=global_vars.MODEL_COMPLEXITY,
        static_image_mode=False,
        enable_segmentation=True
    ) as pose:
        while True:
            try:
                metadata = conn.recv(12)
                if not metadata:
                    break

                global_vars.WIDTH, global_vars.HEIGHT, global_vars.FOCAL_LENGTH = struct.unpack('!III', metadata)
                print(f"Updated settings: WIDTH={global_vars.WIDTH}, HEIGHT={global_vars.HEIGHT}, FOCAL_LENGTH={global_vars.FOCAL_LENGTH}")

                # Receive image size (4 bytes)
                img_size_data = conn.recv(4)
                if not img_size_data:
                    break

                img_size = struct.unpack('!I', img_size_data)[0]

                img_data = bytearray()
                while len(img_data) < img_size:
                    packet = conn.recv(min(4096, img_size - len(img_data)))
                    if not packet:
                        break
                    img_data.extend(packet)

                if len(img_data) != img_size:
                    print("Did not receive the full image. Skipping this frame.")
                    continue
                    
                np_arr = np.frombuffer(img_data, np.uint8)
                image = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
                image = cv2.flip(image, 0)
                image = cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)

                if image is None:
                    print("Failed to decode image. Ensure the image data is correct.")
                    continue

                results = pose.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
                data = ""
                if results.pose_world_landmarks:
                    image_landmarks = results.pose_landmarks
                    world_landmarks = results.pose_world_landmarks

                    model_points = np.float32([[l.x, -l.y, -l.z] for l in world_landmarks.landmark])
                    image_points = np.float32([[l.x * image.shape[1], l.y * image.shape[0]] for l in image_landmarks.landmark])

                    body_world_landmarks_world = compute_real_world_landmarks(model_points, image_points, image.shape)

                    data = "\n".join(
                        [f"FREE|{i}|{body_world_landmarks_world[i][0]}|{body_world_landmarks_world[i][1]}|{body_world_landmarks_world[i][2]}"
                         for i in range(33)]
                    ) #+ "<END>"
                    print(data)
                    conn.sendall(data.encode('utf-8'))
                    print("Sent keypoints data to client.")
            except (BrokenPipeError, ConnectionResetError):
                print(f"Connection lost. Listening for connections on {global_vars.HOST}:{global_vars.PORT}...")
                break

def main():
    broadcast_thread = threading.Thread(target=broadcast_ip, daemon=True)
    broadcast_thread.start()
    
    server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server_socket.bind((global_vars.HOST, global_vars.PORT))
    server_socket.listen(5)
    
    print(f"Listening for connections on {global_vars.HOST}:{global_vars.PORT}...")
    
    while True:
        try:
            client_socket, addr = server_socket.accept()
            print(f"Connection established from {addr[0]}:{addr[1]}")
            process_pose(client_socket)
        except Exception as e:
            print(f"Server error: {e}")
        finally:
            client_socket.close()
            print(f"Connection lost. Listening for connections on {global_vars.HOST}:{global_vars.PORT}...")

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        global_vars.KILL_THREADS = True
        print("Shutting down server...")
    except Exception as e:
        print(f"Unexpected error: {e}")
